# 01_coleta: Coleta dos dados de internações do SIH/SUS (RD) para Minas Gerais, competência 2024

Pipeline de dados de ponta a ponta no Databricks Free Edition.

- Fonte: DATASUS, Sistema de Informações Hospitalares (SIH/SUS), arquivos RD (AIH Reduzida)
- Escopo: estado de Minas Gerais, competência 2024, 12 arquivos mensais (RDMG2401 a RDMG2412)
- Saída: camada Bronze em Parquet no volume dados_mvp, pasta bronze/sih_rd_mg_2024
- Método: PySUS 2.x, função pysus.ftp.sih, download pelo catálogo S3 (Parquet), com filtro do grupo RD
- Controle: coleta idempotente, remove a Bronze antes de recriar, valida volume por mês e trava anti-duplicação

In [0]:
# Garante a versão mais recente do PySUS instalada no cluster.
# A API namespaced (pysus.ftp.sih) só existe na versão 2.x.
%pip install --quiet pysus

In [0]:
import pandas as pd
import pysus
from pyspark.sql.functions import col

# Caminho da camada Bronze no volume (formato Parquet)
caminho_bronze = "/Volumes/workspace/default/dados_mvp/bronze/sih_rd_mg_2024"

# Parâmetros da coleta
UF = "MG"
ANO = 2024
GRUPO = "RD"          # AIH Reduzida, arquivos com prefixo RDMG
MESES = list(range(1, 13))

# Confere se a API namespaced do PySUS 2.x está disponível.
# A API antiga (pysus.online_data) não existe na versão atual, por isso
# a checagem para abortar cedo com uma mensagem clara se estiver desatualizado.
print("Versão do PySUS:", getattr(pysus, "__version__", "desconhecida"))
tem_api_nova = hasattr(pysus, "ftp") and hasattr(pysus.ftp, "sih")
print("API pysus.ftp.sih disponível:", tem_api_nova)
if not tem_api_nova:
    raise SystemExit("PySUS desatualizado. Rode %pip install -U pysus e reinicie o Python.")

In [0]:
# Inspeção prévia dos arquivos remotos, sem baixar nada.
# Lista os arquivos de SIH de MG/2024 e identifica quais são do grupo RD,
# para confirmar o escopo antes da coleta.
bag = pysus.ftp.sih(state=UF, year=ANO, month=MESES, download=False)
print("Arquivos encontrados no catálogo:", len(bag))

# O grupo RD é identificado pelo nome (prefixo RDMG) ou pelo caminho no catálogo (/RD/)
indices_rd = []
for i, f in enumerate(bag):
    nome = str(getattr(f, "name", "")).upper()
    caminho = str(getattr(f, "path", "")).upper()
    eh_rd = nome.startswith("RDMG") or "/RD/" in caminho
    print(f"  [{i}] nome={nome} | RD: {eh_rd}")
    if eh_rd:
        indices_rd.append(i)

print("Total de arquivos RD:", len(indices_rd))

In [0]:
import pandas as pd
import pysus

# Definições locais, a célula é autossuficiente e roda isolada
# sem depender das células anteriores.
caminho_bronze = "/Volumes/workspace/default/dados_mvp/bronze/sih_rd_mg_2024"
UF = "MG"
ANO = 2024

# Idempotência: remove a Bronze antiga antes de recriar do zero.
# Isso garante que reexecutar o notebook não acumule dados de execuções
# anteriores (o append por bloco é seguro porque parte sempre de uma pasta vazia).
dbutils.fs.rm(caminho_bronze, recurse=True)
print("Iniciando coleta limpa dos 12 meses de", ANO)

# Coleta em blocos de 3 meses, para não estourar a memória do cluster
blocos = [[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12]]

for bloco in blocos:
    # Lista os arquivos remotos do SIH de MG/2024 no bloco, sem baixar
    bag = pysus.ftp.sih(state=UF, year=ANO, month=bloco, download=False)

    # Filtra apenas o grupo RD (AIH Reduzida).
    # A função pysus.ftp.sih lista todos os grupos do SIH (RD, RJ, ER e SP);
    # sem o filtro, o volume baixado infla de ~380 mil para ~5,4 milhões de
    # linhas por bloco, estourando o limite do Spark
    # (erro LOCAL_RELATION_SIZE_LIMIT_EXCEEDED e timeout de IOStream).
    # O DATASUS nomeia os arquivos como RDMG<YYMM>, então o prefixo RDMG
    # seleciona exatamente os arquivos de RD de Minas Gerais.
    indices_rd = []
    for i, f in enumerate(bag):
        nome = str(getattr(f, "name", "")).upper()
        caminho = str(getattr(f, "path", "")).upper()
        if nome.startswith("RDMG") or "/RD/" in caminho:
            indices_rd.append(i)

    if not indices_rd:
        raise SystemExit(f"Nenhum arquivo RD encontrado no bloco {bloco}. Abortando.")

    print("Bloco", bloco, "- arquivos RD:", len(indices_rd))

    # Baixa apenas os arquivos RD do bloco e concatena em um DataFrame pandas
    local = bag.download(indexes=indices_rd)
    df_mes = local.to_dataframe()
    print("Bloco", bloco, "- linhas totais (RD):", len(df_mes))

    # Verificação de volume por mês dentro do bloco.
    # O esperado é 120 a 140 mil linhas por mês; valores zerados ou muito
    # baixos indicam download truncado ou arquivo ausente, e paramos ali.
    print(df_mes.groupby("MES_CMPT").size())

    # Metadados de controle da camada Bronze
    df_mes["dt_ingestao"] = pd.Timestamp.now()
    df_mes["fonte"] = "DATASUS/SIH"
    df_mes["uf"] = UF
    df_mes["ano_competencia"] = ANO

    # Converte para Spark e grava em Parquet em pedaços.
    # A gravação em pedaços evita o limite de tamanho da relação local do
    # Spark (erro LOCAL_RELATION_SIZE_LIMIT_EXCEEDED) para blocos grandes.
    chunk_size = 150000
    for inicio in range(0, len(df_mes), chunk_size):
        fatia = df_mes.iloc[inicio:inicio + chunk_size]
        df_spark = spark.createDataFrame(fatia)
        df_spark.write.mode("append").format("parquet").save(caminho_bronze)
    print("Bloco", bloco, "salvo na Bronze:", len(df_mes), "linhas")

print("Coleta concluída.")

In [0]:
from pyspark.sql.functions import col

# Validação da camada Bronze após a coleta completa
caminho_bronze = "/Volumes/workspace/default/dados_mvp/bronze/sih_rd_mg_2024"
df_bronze = spark.read.parquet(caminho_bronze)

# Total de registros e colunas
print("TOTAL de linhas na Bronze:", df_bronze.count())
print("Total de colunas:", len(df_bronze.columns))

# Distribuição por competência: devem aparecer as 12 competências de 2024,
# com volume parecido (120 a 140 mil por mês)
print("Distribuição por competência:")
df_bronze.groupBy("ANO_CMPT", "MES_CMPT").count().orderBy("MES_CMPT").show(15, truncate=False)

# Completude: nulos nas colunas-chave
chaves = ["N_AIH", "ANO_CMPT", "MES_CMPT", "DT_INTER", "DT_SAIDA", "DIAG_PRINC", "SEXO", "IDADE"]
for chave in chaves:
    nulos = df_bronze.filter(col(chave).isNull()).count()
    print("Nulos em", chave, ":", nulos)

# Unicidade: chaves duplicadas por competência + N_AIH.
# Um número pequeno é esperado, pois o DATASUS reprocessa AIH e o arquivo
# original traz repetições; um número alto indica append acumulado de reexecuções.
duplicadas = df_bronze.groupBy("ANO_CMPT", "MES_CMPT", "N_AIH").count().filter(col("count") > 1).count()
print("Chaves duplicadas (competência + N_AIH):", duplicadas)

# Trava anti-duplicação: se o número de duplicadas passar do padrão natural
# da fonte, aborta para não seguir com a Bronze corrompida por reexecuções.
if duplicadas > 1000:
    raise SystemExit("Bronze possivelmente duplicada por execuções repetidas. Rode o notebook inteiro de novo, a célula 5 remove e recria a Bronze.")

# Estrutura dos metadados de controle
print("Schema dos metadados de controle:")
df_bronze.select("dt_ingestao", "fonte", "uf", "ano_competencia").printSchema()